# Combining Analyses

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.9.0

**Author**: Mateusz Jakub Bała

## About

Some questions are two questions at once: the prompt gammas of ortho-positronium
decays, or the events that hold either a prompt gamma or a Compton scattering.
Analyses combine with `&` and `|`, and one analysis continues another with
`then`.

`&` and `|` perform each analysis on its own and combine what they kept;
`then` performs the second on what the first kept. Most of the time the two
ways agree. This notebook shows how each works, why `&` and `|` can be grouped
in any way, and the case in which `&` and `then` part ways.

## What will you learn from this notebook

- keep what two analyses both keep, with `&`, and what either keeps, with `|`
- group combinations in any way, and know why the result does not change
- continue an analysis with `then`, and see where the second one starts
- recognise when `&` and `then` give different answers

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.8.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file carried in the repository, described in
[`../data/README.md`](../data/README.md): 500 hits of a scene with a
`PositroniumSource` configured with every channel at once, beside a
back-to-back source. Nothing is written, so no output directory is needed.

In [2]:
from opengate_gate_tree import (
    EventTrackHit,
    EventTrackHitAnalysis,
    EventTrackHitField,
    GammaType,
    GateTree,
    Process,
    SourceType,
    read_tree,
)

Analysis = EventTrackHitAnalysis
Field = EventTrackHitField
data = read_tree(paths.input_example("positronium-source-all-variants.root"), GateTree.HITS)
representation = EventTrackHit(data)
representation

EventTrackHit(events=115, tracks=311, hits=500)

### `&` Keeps What Both Keep

Two analyses asking about different levels: the events emitted by
ortho-positronium, and the prompt gammas.

In [3]:
ortho = Analysis().source_type(SourceType.ORTHO_POSITRONIUM)
prompt = Analysis().tracks().gamma_type(GammaType.PROMPT)

both = ortho & prompt
both, both.apply(representation)

((EventTrackHitAnalysis().source_type(SourceType.ORTHO_POSITRONIUM) & EventTrackHitAnalysis().tracks().gamma_type(GammaType.PROMPT)),
 EventTrackHit(events=19, tracks=19, hits=47))

19 prompt gammas, one in each ortho-positronium decay with de-excitation. Each
analysis was performed on its own, from the events, and the events, tracks and
hits both kept are what is left. The prompt gammas of para-positronium decays
were kept by `prompt` but not by `ortho`, so they are not.

### `|` Keeps What Either Keeps

In [4]:
either = ortho | prompt
either.apply(representation)

EventTrackHit(events=62, tracks=183, hits=277)

Every track of the 51 ortho-positronium decays, and the prompt gammas of the
other eleven events that hold one.

### Grouping Does Not Matter

Both operators work on sets of events, tracks and hits, so they are associative
and commutative. The second example of the analysis concept relies on it: three
analyses joined with `&` mean the same however they are grouped.

In [5]:
compton = Analysis().hits().process(Process.COMPTON)

grouped_left = ((ortho & prompt) & compton).apply(representation)
grouped_right = (ortho & (prompt & compton)).apply(representation)
swapped = (compton & ortho & prompt).apply(representation)

grouped_left.hit_count == grouped_right.hit_count == swapped.hit_count

True

A combination prints with its brackets, so it reads as it was grouped:

In [6]:
(ortho & prompt) | compton

((EventTrackHitAnalysis().source_type(SourceType.ORTHO_POSITRONIUM) & EventTrackHitAnalysis().tracks().gamma_type(GammaType.PROMPT)) | EventTrackHitAnalysis().hits().process(Process.COMPTON))

### `then` Performs One Analysis On What Another Kept

`then` appends the operations of the second analysis to the first. The second
starts at the events, as it would on its own: when the first ends elsewhere,
`then` puts `events()` between them, and the printed chain shows it.

In [7]:
Analysis().track(1).gamma_type(GammaType.PROMPT).then(Analysis().hits(2))

EventTrackHitAnalysis().track(1).gamma_type(GammaType.PROMPT).events().hits(2)

### Where `&` And `then` Part Ways

They agree whenever no count or position of one side follows a filter of the
other. When one does, they ask different questions:

In [8]:
annihilation = Analysis().tracks().gamma_type(GammaType.ANNIHILATION)
three = Analysis().tracks(3)

(annihilation & three).apply(representation), annihilation.then(three).apply(representation)

(EventTrackHit(events=43, tracks=118, hits=173),
 EventTrackHit(events=51, tracks=153, hits=205))

`annihilation & three` keeps the events of three gammas - performed on its own,
`three` counts every gamma - and of them the annihilation gammas: the
ortho-positronium decays without a prompt gamma and the para-positronium decays
with one, 43 events. `annihilation.then(three)` counts the annihilation gammas
the first analysis kept: the 51 events with three of them, every
ortho-positronium decay, prompt gamma or not.

Only analyses of the same kind combine:

In [9]:
try:
    ortho & "a string"
except TypeError as error:
    print(error)

EventTrackHitAnalysis combines only with another EventTrackHitAnalysis through '&', not with str.


## Summary

`&` and `|` perform two analyses on their own and keep what both, or either,
kept; they can be grouped in any way. `then` performs the second analysis on
what the first kept, starting at the events. The two ways agree unless a count
or a position of one side follows a filter of the other.

## Bibliography

- [Analysis Of Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/analysis.html)
  — combining analyses with `&`, `|` and `then`
- [Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/event-track-hit.html)
  — the representation an analysis is performed on, and the order it counts in
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — what `sourceType`, `decayType` and `gammaType` mean

## Changelog

- v1: written.